# 🎬 AutoShorts - AI-Powered Viral Shorts Generator on Google Colab

Turn long-form YouTube videos, streams, or local footage into viral 9:16 vertical shorts with:
- 🧠 **AI Moment Detection** (Gemini 2.0 / OpenAI long-context video understanding)
- ⚡ **GPU-Accelerated Processing** (CUDA decoding, CuPy framing & 9:16 cropping, NVENC encoding)
- 🎨 **Kinetic Subtitles** (OpenAI Whisper + PyCaps styled animated captions)
- 🗣️ **Optional AI Voiceover Narration** (Qwen3-TTS)
- 📥 **Direct YouTube / Stream Link Ingestion** via `yt-dlp`

### Step 1: Check GPU Acceleration
Ensure you are running on a **GPU runtime** (Edit -> Notebook settings -> Hardware accelerator -> T4 GPU).

In [ ]:
!nvidia-smi

### Step 2: Clone Repository & Install Dependencies

In [ ]:
import os
# If not already inside the repository, clone your fork:
# !git clone https://github.com/YOUR_USERNAME/autoshorts.git
# %cd autoshorts

!apt-get update -qq && apt-get install -y -qq ffmpeg sox libsox-fmt-all build-essential

# Install PyTorch with CUDA 12.8 & TorchCodec
!pip install --upgrade pip wheel setuptools
!pip install torch==2.10.0 torchvision==0.25.0 torchaudio==2.10.0 --index-url https://download.pytorch.org/whl/cu128
!pip install --no-deps torchcodec==0.10.0

# Install core requirements and yt-dlp
!pip install -r requirements.txt

# Install Playwright browser for PyCaps styled subtitles
!playwright install chromium --with-deps

# Build Decord with CUDA support
!make install_decord || pip install decord

### Step 3: Configure AI Provider & API Keys

In [ ]:
#@title Configure AutoShorts Environment {.run: "auto"}
GEMINI_API_KEY = "" #@param {type:"string"}
OPENAI_API_KEY = "" #@param {type:"string"}
AI_PROVIDER = "gemini" #@param ["gemini", "openai", "local"]
VIDEO_TYPE = "gaming" #@param ["gaming", "esports", "podcasts", "interviews", "news_commentary", "entertainment", "tv_shows", "comedy", "music", "vlogs", "cooking_diy", "fitness", "educational", "documentaries", "sports"]
CAPTION_STYLE = "auto" #@param {type:"string"}
ENABLE_SUBTITLES = True #@param {type:"boolean"}
ENABLE_TTS = False #@param {type:"boolean"}
SCENE_LIMIT = 4 #@param {type:"integer"}

env_content = f"""AI_PROVIDER={AI_PROVIDER}
GEMINI_API_KEY={GEMINI_API_KEY}
OPENAI_API_KEY={OPENAI_API_KEY}
VIDEO_TYPE={VIDEO_TYPE}
CAPTION_STYLE={CAPTION_STYLE}
ENABLE_SUBTITLES={str(ENABLE_SUBTITLES).lower()}
SUBTITLE_MODE=speech
ENABLE_TTS={str(ENABLE_TTS).lower()}
TTS_LANGUAGE=en
SCENE_LIMIT={SCENE_LIMIT}
TARGET_RATIO_W=9
TARGET_RATIO_H=16
DEBUG_SKIP_ANALYSIS=false
DEBUG_SKIP_RENDER=false
"""

with open(".env", "w", encoding="utf-8") as f:
    f.write(env_content)

print("✅ Configuration saved to .env")

### Step 4: Generate Shorts from YouTube Video or Stream
Paste any YouTube video or livestream VOD URL below and run the cell.

In [ ]:
#@title Ingest YouTube URL and Generate Shorts
YOUTUBE_URL = "https://www.youtube.com/watch?v=..." #@param {type:"string"}

assert YOUTUBE_URL and "http" in YOUTUBE_URL, "Please provide a valid YouTube URL"

!python run.py --input "{YOUTUBE_URL}" --output generated/

### Step 5: Download Generated Clips
Download all generated short clips directly to your computer or copy to Google Drive.

In [ ]:
from google.colab import files
import shutil
from pathlib import Path

generated_dir = Path("generated")
clips = list(generated_dir.glob("*.mp4"))

if not clips:
    print("No clips found in generated/")
else:
    print(f"Found {len(clips)} generated clip(s):")
    for c in clips:
        print(f" - {c.name} ({c.stat().st_size / (1024*1024):.1f} MB)")

    # Zip and download
    shutil.make_archive("autoshorts_clips", "zip", "generated")
    files.download("autoshorts_clips.zip")
    print("📦 Download started for autoshorts_clips.zip")

### (Optional) Step 6: Launch Streamlit Web UI via Cloudflare Tunnel

In [ ]:
# Download Cloudflare Tunnel for quick public URL access
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

# Start Streamlit in background
import subprocess
subprocess.Popen(["streamlit", "run", "src/dashboard/About.py", "--server.port", "8501", "--server.headless", "true"])

# Expose via Cloudflare Tunnel
!cloudflared tunnel --url http://localhost:8501